# MPIPS Neural-Calibrated Radiography Batch

This Colab notebook installs MPIPS from a private, pinned GitHub revision, accepts public Google Drive links or private mounted paths, builds or reuses a validated neural dot-grid calibration, and processes one or many radiograph NPZ files into 16-bit TIFFs.

> Security: NPZ inputs contain pickled Python dictionaries. Use only trusted Madeena files. Public Drive URLs must be shared as **Anyone with the link**; use mounted paths for private data.

In [ ]:
# 1. Force the Google Drive mount to flush its cache and sync
# from google.colab import drive
# drive.flush_and_unmount()

# 2. Re-mount the drive
# drive.mount('/content/drive')


In [ ]:
#@title 1. Install MPIPS from the private repository
# import base64
# import os
# import subprocess
# import sys
# from google.colab import userdata

# MPIPS_GIT_REF = "6350d5e0af3c46f554073e42f26cfa5ff842faa0" #@param {type:"string"}
# if len(MPIPS_GIT_REF) != 40 or any(c not in "0123456789abcdefABCDEF" for c in MPIPS_GIT_REF):
#     raise ValueError("MPIPS_GIT_REF must be the 40-character implementation commit SHA.")
# token = userdata.get("GITHUB_TOKEN")
# if not token:
#     raise RuntimeError("Add a read-only GITHUB_TOKEN in Colab Secrets before continuing.")

# credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
# install_env = os.environ.copy()
# install_env.update({
#     "GIT_CONFIG_COUNT": "1",
#     "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
#     "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}",
# })
# requirement = (
#     "mpips[colab] @ git+https://github.com/Madeena-software/mpips.git"
#     f"@{MPIPS_GIT_REF}"
# )
# subprocess.run(
#     [sys.executable, "-m", "pip", "install", "--upgrade", requirement],
#     check=True, env=install_env,
# )
# del token, credential, install_env
# print("MPIPS installation completed.")

In [ ]:
#@title 2. Mount Drive and configure inputs
from pathlib import Path
# from google.colab import drive

# drive.mount("/content/drive")

CALIBRATION_SOURCE = "/var/www/mpips/research/kambing-260714/data/kalibrasi-gotri/BED_1783219960026.npz" #@param {type:"string"}
CALIBRATION_GAIN_SOURCE = "" #@param {type:"string"}
GAIN_SOURCES = "/var/www/mpips/research/kambing-260714/data/gain/BED_1783219207291.npz" #@param {type:"string"}
GOAT_SOURCES = "/var/www/mpips/research/kambing-260714/data/kambing/BED_1783222264263.npz" #@param {type:"string"}
OUTPUT_ROOT = "/var/www/mpips/research/kambing-260714/data/output" #@param {type:"string"}

# CALIBRATION_GAIN_SOURCE optionally selects exactly one gain NPZ.
# All source fields accept mounted paths or public Drive URLs; GAIN_SOURCES
# and GOAT_SOURCES also accept folders or newline-separated entries.
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
WORK_ROOT = Path("./data/work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Durable outputs: {OUTPUT_ROOT}")

In [ ]:
# 3. Resolve and validate input locations
import shutil
from mpips.workflows.imager_pipeline import resolve_npz_sources

input_workspace = WORK_ROOT / "inputs"
if input_workspace.exists():
    shutil.rmtree(input_workspace)
input_workspace.mkdir(parents=True)

calibration_files = resolve_npz_sources(
    CALIBRATION_SOURCE, input_workspace / "calibration"
)
if len(calibration_files) != 1:
    raise ValueError(f"Exactly one calibration NPZ is required; found {len(calibration_files)}")
calibration_gain_files = []
if CALIBRATION_GAIN_SOURCE.strip():
    calibration_gain_files = resolve_npz_sources(
        CALIBRATION_GAIN_SOURCE, input_workspace / "calibration-gain"
    )
    if len(calibration_gain_files) != 1:
        raise ValueError(
            "Exactly one calibration gain NPZ is required when provided; "
            f"found {len(calibration_gain_files)}"
        )
gain_files = resolve_npz_sources(GAIN_SOURCES, input_workspace / "gains")
goat_files = resolve_npz_sources(GOAT_SOURCES, input_workspace / "goats")
print(f"Calibration: {calibration_files[0].name}")
print(
    "Calibration gain:",
    calibration_gain_files[0].name
    if calibration_gain_files
    else "stored processedimage",
)
print(f"Gains: {len(gain_files)} NPZ file(s)")
print(f"Radiographs: {len(goat_files)} NPZ file(s)")

In [ ]:
#@title 4. Advanced calibration and pipeline settings
from mpips.workflows.imager_pipeline import (
    NeuralCalibrationConfig, ImagerPipelineConfig,
)

FORCE_RETRAIN = False #@param {type:"boolean"}
TRAINING_EPOCHS = 5000 #@param {type:"integer"}
DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
REMAP_STEP = 4 #@param {type:"slider", min:1, max:8, step:1}
CANVAS_MODE = "expanded" #@param ["fixed", "expanded"]
EXPANDED_MARGIN = 16 #@param {type:"integer"}
WAVELET = "sym4" #@param ["sym4", "db1", "haar"]
WAVELET_LEVEL = 3 #@param {type:"slider", min:1, max:5, step:1}
THRESHOLD_METHOD = "auto" #@param ["auto", "none"]
CONTRAST_SATURATED_PIXELS = 5.0 #@param {type:"number"}
CLAHE_BLOCKSIZE = 127 #@param {type:"integer"}
CLAHE_MAX_SLOPE = 0.6 #@param {type:"number"}
CLAHE_FAST = False #@param {type:"boolean"}
USE_MEDIAN_FILTER = True #@param {type:"boolean"}
MEDIAN_FILTER_RADIUS = 2 #@param {type:"slider", min:1, max:3, step:1}

calibration_config = NeuralCalibrationConfig(
    epochs=TRAINING_EPOCHS, device=DEVICE, remap_step=REMAP_STEP,
    canvas_mode=CANVAS_MODE, expanded_margin=EXPANDED_MARGIN,
    force_retrain=FORCE_RETRAIN,
)
pipeline_config = ImagerPipelineConfig(
    wavelet=WAVELET, wavelet_level=WAVELET_LEVEL,
    threshold_method=THRESHOLD_METHOD,
    contrast_saturated_pixels=CONTRAST_SATURATED_PIXELS,
    clahe_blocksize=CLAHE_BLOCKSIZE, clahe_max_slope=CLAHE_MAX_SLOPE,
    clahe_fast=CLAHE_FAST,
    use_median_filter=USE_MEDIAN_FILTER,
    median_filter_radius=MEDIAN_FILTER_RADIUS,
)
print(calibration_config)
print(pipeline_config)

In [ ]:
# 5. Build or reuse the validated neural calibration
import json
from mpips.workflows.imager_pipeline import build_or_load_calibration

calibration = build_or_load_calibration(
    calibration_files[0],
    Path(OUTPUT_ROOT) / "calibration-cache",
    calibration_config,
    calibration_gain_npz=(
        calibration_gain_files[0] if calibration_gain_files else None
    ),
)
metrics = json.loads(calibration.metrics_path.read_text())
print("Calibration cache hit:", calibration.cache_hit)
print("Calibration fingerprint:", calibration.fingerprint)
print(json.dumps(metrics["reductions_percent"], indent=2))
print("Artifacts:", calibration.directory)

In [ ]:
# 6. Process the radiograph batch
from mpips.workflows.imager_pipeline import load_gain_catalog, process_npz_batch

gains = load_gain_catalog(gain_files)
def show_progress(current, total, item):
    marker = "OK" if item.status == "completed" else "FAILED"
    print(f"[{current}/{total}] {marker}: {Path(item.source).name}")
    if item.error:
        print("   ", item.error)

batch = process_npz_batch(
    goat_files, gains, calibration, Path(OUTPUT_ROOT), pipeline_config,
    on_progress=show_progress,
)
print(f"Completed: {batch.succeeded}; failed: {batch.failed}")
print("Manifest:", batch.manifest_path)

In [ ]:
# 7. Visualize the first successful result
%matplotlib inline
import cv2
import matplotlib.pyplot as plt
import numpy as np

successful = next((item for item in batch.items if item.status == "completed"), None)
if successful is None:
    print("No successful image is available to display. See the manifest for errors.")
else:
    with np.load(successful.source, allow_pickle=True) as source_npz:
        raw_image = source_npz["rawimage"]
    processed_image = cv2.imread(successful.output, cv2.IMREAD_UNCHANGED)
    figure, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(raw_image, cmap="gray")
    axes[0].set_title("Raw radiograph")
    axes[1].imshow(processed_image, cmap="gray")
    axes[1].set_title("Neural-calibrated MPIPS result")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    print(successful.output)